In [1]:
from dotenv import load_dotenv

load_dotenv()

import ast
import operator as op
import os

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

In [ ]:
# Allowed operators
_OPS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.FloorDiv: op.floordiv,
    ast.Mod: op.mod,
    ast.Pow: op.pow,
    ast.UAdd: op.pos,
    ast.USub: op.neg,
}


def _eval_node(node) -> int | float:
    # Python 3.8+ uses Constant for numbers
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    if isinstance(node, ast.Expr):
        return _eval_node(node.value)
    raise ValueError(
        "Unsupported expression. Use numbers, + - * / // % ** and parentheses only."
    )


def safe_calc(expr: str) -> float:
    expr = expr.strip()
    if len(expr) > 200:
        raise ValueError("Expression too long.")
    tree = ast.parse(expr, mode="eval")
    return float(_eval_node(tree.body))


@tool
def calculate(expression: str) -> str:
    """Safely evaluate arithmetic like '(12345*6789)+98765'. Returns only the number."""
    try:
        val = safe_calc(expression)
        if abs(val) > 1e18:
            return "Error: result too large."
        # Pretty output: drop trailing .0 if it's an integer
        return str(int(val)) if val.is_integer() else str(val)
    except Exception as e:
        return f"Error: {e}"


def build_calc_agent():
    llm = ChatOpenAI(
        model=os.getenv("GPT_MODEL", "your-gpt-model"),
        base_url=os.getenv("PORTKEY_BASE_URL", "your-base-url"),
        api_key=os.getenv("PORTKEY_API_KEY", "your-api-key"),
    )
    tools = [calculate]
    SYSTEM_PROMPT = (
        "You are a precise math assistant.\n"
        "RULES:\n"
        "1) Always use the `calculate` tool for arithmetic; do not compute in your head.\n"
        "2) Convert natural language to a clean math expression.\n"
        "   - If user writes '^' for exponent, convert to '**'.\n"
        "3) If user asks for 'only the number', return only the number.\n"
        "4) Use only digits, + - * / // % ** ( ) and decimal points."
    )

    return create_agent(llm, tools=tools, system_prompt=SYSTEM_PROMPT)


calc_agent = build_calc_agent()


def ask(agent, query: str) -> str:
    return agent.invoke({"messages": [("human", query)]})["messages"][-1].content


print(" Calculator agent ready.")

 Calculator agent ready.


In [4]:
print(ask(calc_agent, "Compute (12345*6789)+98765. Give only the number."))

83908970


In [ ]:
# Plain LLM (no tool) for comparison
llm_plain = ChatOpenAI(
    model=os.getenv("GPT_MODEL", "your-gpt-model"),
    base_url=os.getenv("PORTKEY_BASE_URL", "your-base-url"),
    api_key=os.getenv("PORTKEY_API_KEY", "your-api-key"),
)

q1 = "What is 234 * 19? Give only the number."
q2 = "Compute (12345*6789)+98765. Give only the number."
q3 = "What is 13.7% of 12345 plus 98.76^2? Round to 2 decimals."

print("— LLM only —")
print("Q1:", llm_plain.invoke(q1).content)
print("Q2:", llm_plain.invoke(q2).content)
print("Q3:", llm_plain.invoke(q3).content)

print("\n— Agent with calculator —")
print("Q1:", ask(calc_agent, q1))
print("Q2:", ask(calc_agent, q2))
print("Q3:", ask(calc_agent, q3))

— LLM only —
Q1: 4446
Q2: 83908970
Q3: Compute:
- 13.7% of 12345 = 12345 × 0.137 = 1691.265
- 98.76^2 = 9753.5376

Sum = 1691.265 + 9753.5376 = 11444.8026

Rounded to 2 decimals: 11,444.80.

— Agent with calculator —
Q1: 4446
Q2: 83908970
Q3: 11444.80
